In [2]:
# 1.
import pandas as pd
df = pd.read_csv("downloads/data/data/titanic.csv")
print(df.head(3))
print(df.shape)
print(df.select_dtypes(include = "str").columns)

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
(891, 12)
Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='str')


In [3]:
#2.
print(df[['Age','Fare']].describe())
print()
print(df['Age'].median())
print()
print(df['Fare'].mean())
print()
print(df['Fare'].max())

              Age        Fare
count  714.000000  891.000000
mean    29.699118   32.204208
std     14.526497   49.693429
min      0.420000    0.000000
25%     20.125000    7.910400
50%     28.000000   14.454200
75%     38.000000   31.000000
max     80.000000  512.329200

28.0

32.204207968574636

512.3292


age count differ from no. of rows because some rows has null value for age 

In [4]:
#3.
print(type(df['Age']))
print(type(df[['Age']]))
data = df[['Name','Sex','Age','Fare']]
df1 = pd.DataFrame(data)
print()
print(df1.shape)

<class 'pandas.Series'>
<class 'pandas.DataFrame'>

(891, 4)


In [5]:
#4.
count = df[(df['Sex'] == 'female') & (df['Pclass'] == 1)]
print(count.shape[0])
print(count["Survived"].sum())
passenger = df[(df['Age'] < 10) | (df['Fare'] > 200)]
print(passenger.shape[0])

94
91
82


In [6]:
#5.
print(df.loc[10:15,['Name','Fare']])
print()
print(df.iloc[10:15,[3,9]])

                                    Name     Fare
10       Sandstrom, Miss. Marguerite Rut  16.7000
11              Bonnell, Miss. Elizabeth  26.5500
12        Saundercock, Mr. William Henry   8.0500
13           Andersson, Mr. Anders Johan  31.2750
14  Vestrom, Miss. Hulda Amanda Adolfina   7.8542
15      Hewlett, Mrs. (Mary D Kingcome)   16.0000

                                    Name     Fare
10       Sandstrom, Miss. Marguerite Rut  16.7000
11              Bonnell, Miss. Elizabeth  26.5500
12        Saundercock, Mr. William Henry   8.0500
13           Andersson, Mr. Anders Johan  31.2750
14  Vestrom, Miss. Hulda Amanda Adolfina   7.8542


loc return 6 rows and iloc return 5 rows because loc uses labels, while iloc uses integer positions.

In [7]:
#6.
df.sort_values("Fare", ascending=False)[["Name", "Pclass", "Fare"]].head(5)

,Name,Pclass,Fare
679,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292
258,"Ward, Miss. Anna",1,512.3292
737,"Lesurer, Mr. Gustave J",1,512.3292
88,"Fortune, Miss. Mabel Helen",1,263.0000
438,"Fortune, Mr. Mark",1,263.0000


all highest fare are from 1st class

In [8]:
#7
print(df.groupby("Sex")["Survived"].mean())
by_sex_class = df.groupby(["Sex", "Pclass"])["Survived"].mean()
print(by_sex_class)

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64


Best: Female passengers in 1st class, with a survival rate of approximately 96.8%.
Worst: Male passengers in 3rd class, with a survival rate of approximately 13.5%.

In [9]:
#8
class_summary = df.groupby('Pclass').agg(passengers=("PassengerId", "count"),
                                       survival_rate=("Survived", "mean"),
                                       avg_fare=("Fare", "mean"),
                                       oldest=("Age", "max"))
print(class_summary)

        passengers  survival_rate   avg_fare  oldest
Pclass                                              
1              216       0.629630  84.154687    80.0
2              184       0.472826  20.662183    70.0
3              491       0.242363  13.675550    74.0


In [10]:
#9
df["fare_vs_class"] = df["Fare"] - df.groupby("Pclass")["Fare"].transform("mean")
df.loc[df["fare_vs_class"].idxmax(), ["Name", "Pclass", "Fare", "fare_vs_class"]]

Name             Ward, Miss. Anna
Pclass                          1
Fare                     512.3292
fare_vs_class          428.174512
Name: 258, dtype: object

transform() returns results aligned with the original rows, so you can assign them directly to df. agg() returns one aggregated result per group, so it produces only one average per passenger class rather than one average per passenger.

In [11]:
#10
raw = df.copy()

missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percent })
print(missing_summary)

               Missing_Count  Missing_Percentage
PassengerId                0            0.000000
Survived                   0            0.000000
Pclass                     0            0.000000
Name                       0            0.000000
Sex                        0            0.000000
Age                      177           19.865320
SibSp                      0            0.000000
Parch                      0            0.000000
Ticket                     0            0.000000
Fare                       0            0.000000
Cabin                    687           77.104377
Embarked                   2            0.224467
fare_vs_class              0            0.000000


i would drop 'Cabin' because a  large proportion of its values are missing

In [13]:
#11
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
age_original = df["Age"].copy()
# Method 1: Fill with overall median
age_overall = age_original.fillna(age_original.median())
# Method 2: Fill with median for each Sex + Pclass group
age_group = age_original.fillna(
    df.groupby(["Sex", "Pclass"])["Age"].transform("median"))
women_first = (df["Sex"] == "female") & (df["Pclass"] == 1)

true_mean = age_original[women_first].mean()
overall_mean = age_overall[women_first].mean()
group_mean = age_group[women_first].mean()

print("True mean before filling:", true_mean)
print("Mean after overall median:", overall_mean)
print("Mean after group median:", group_mean)
df["Age"] = age_group

True mean before filling: 34.61176470588235
Mean after overall median: 33.97872340425532
Mean after group median: 34.648936170212764


In [14]:
#12
df["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.")
print(df["Title"].nunique())
print(df["Title"].value_counts())
df["Title"] = df["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"})
df["Title"] = df["Title"].where(
    df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]),
    "Rare")
print(df["Title"].value_counts())
print("Distinct standardized titles:", df["Title"].nunique())

17
Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64
Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64
Distinct standardized titles: 5


In [15]:
#13
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = df["FamilySize"] == 1
def label_age(age):
    if age < 13:
        return "child"
    elif age < 18:
        return "teen"
    elif age < 60:
        return "adult"
    else:
        return "senior"

df["AgeGroup"] = df["Age"].apply(label_age)
survival_alone = df.groupby("IsAlone")["Survived"].mean()
print(survival_alone)

IsAlone
False    0.505650
True     0.303538
Name: Survived, dtype: float64


In [16]:
#14
print(df.duplicated().sum())
shared_ticket = df["Ticket"].duplicated(keep=False)
print(shared_ticket.sum())
cols = ["Sex", "Embarked", "Title", "AgeGroup"]
memory_before = df[cols].memory_usage(deep=True).sum()
for col in cols:
    df[col] = df[col].astype("category")
memory_after = df[cols].memory_usage(deep=True).sum()
print("Memory before:", memory_before, "bytes")
print("Memory after:", memory_after, "bytes")
print("Memory saved:", memory_before - memory_after, "bytes")

0
344
Memory before: 40648 bytes
Memory after: 3864 bytes
Memory saved: 36784 bytes


In [17]:
#15
ports = pd.DataFrame({
    "Embarked": ["S", "C", "Q", "B"],
    "Port": ["Southampton", "Cherbourg", "Queenstown", "Belfast"],
    "Country": ["England", "France", "Ireland", "UK"]})
inner = raw.merge(ports, on="Embarked", how="inner")
left = raw.merge(ports, on="Embarked", how="left")
right = raw.merge(ports, on="Embarked", how="right")
outer = raw.merge(ports, on="Embarked", how="outer", indicator=True)
print("Inner:", inner.shape[0])
print("Left:", left.shape[0])
print("Right:", right.shape[0])
print("Outer:", outer.shape[0])
print(outer["_merge"].value_counts())

Inner: 889
Left: 891
Right: 890
Outer: 892
_merge
both          889
left_only       2
right_only      1
Name: count, dtype: int64


In [18]:
#16
first = df[df["Pclass"] == 1]
third = df[df["Pclass"] == 3]

stacked = pd.concat(
    [first, third],
    keys=["1st", "3rd"])
print(stacked.shape)
print(stacked.head())
port_survival = df.groupby("Embarked")["Survived"].mean()
port_count = df.groupby("Embarked").size()

port_summary = pd.concat(
    [port_survival, port_count],
    axis=1,
    keys=["survival_rate", "passenger_count"]
)
print(port_summary)

(707, 17)
        PassengerId  Survived  Pclass  \
1st 1             2         1       1   
    3             4         1       1   
    6             7         0       1   
    11           12         1       1   
    23           24         1       1   

                                                     Name     Sex   Age  \
1st 1   Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0   
    3        Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0   
    6                             McCarthy, Mr. Timothy J    male  54.0   
    11                           Bonnell, Miss. Elizabeth  female  58.0   
    23                       Sloper, Mr. William Thompson    male  28.0   

        SibSp  Parch    Ticket     Fare Cabin Embarked  fare_vs_class Title  \
1st 1       1      0  PC 17599  71.2833   C85        C     -12.871387   Mrs   
    3       1      0    113803  53.1000  C123        S     -31.054687   Mrs   
    6       0      0     17463  51.8625   E46        S 

In [19]:
#17
survival_pivot = pd.pivot_table(df,values="Survived",
                                index="Sex",
                                columns="Pclass",
                                aggfunc="mean",
                                 margins=True)

print(survival_pivot)
class_survival = pd.crosstab(df["Pclass"],df["Survived"],normalize="index") * 100
print(class_survival)

Pclass         1         2         3       All
Sex                                           
female  0.968085  0.921053  0.500000  0.742038
male    0.368852  0.157407  0.135447  0.188908
All     0.629630  0.472826  0.242363  0.383838
Survived          0          1
Pclass                        
1         37.037037  62.962963
2         52.717391  47.282609
3         75.763747  24.236253


In [20]:
#18
title_class_count = pd.pivot_table(df,values="PassengerId",index="Title",columns="Pclass",aggfunc="count")
print(title_class_count)
title_class_count = title_class_count.fillna(0)
age_pivot = pd.pivot_table(df,values="Age",index="Title",columns="Pclass",aggfunc="mean")
print(age_pivot)

Pclass      1     2      3
Title                     
Master    3.0   9.0   28.0
Miss     48.0  35.0  102.0
Mr      107.0  91.0  319.0
Mrs      43.0  41.0   42.0
Rare     15.0   8.0    NaN
Pclass          1          2          3
Title                                  
Master   5.306667   2.258889   8.157857
Miss    29.854167  22.871429  17.862745
Mr      41.285047  32.494505  27.673981
Mrs     39.395349  33.682927  30.940476
Rare    47.066667  42.000000        NaN


In [21]:
#19
class_long = class_summary.reset_index().melt(id_vars="Pclass",
                            value_vars=["survival_rate", "avg_fare"],
                            var_name="Metric",
                            value_name="Value")
print(class_long)
print(class_long.shape)
wide = by_sex_class.unstack()
print(wide)
back = wide.stack()
print(back)
print(by_sex_class.equals(back))

   Pclass         Metric      Value
0       1  survival_rate   0.629630
1       2  survival_rate   0.472826
2       3  survival_rate   0.242363
3       1       avg_fare  84.154687
4       2       avg_fare  20.662183
5       3       avg_fare  13.675550
(6, 3)
Pclass         1         2         3
Sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447
Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
dtype: float64
True


In [22]:
#20
group_summary = df.groupby(["Sex", "Pclass", "AgeGroup"],observed=True).agg(
    passengers=("Survived", "count"),
    survival_rate=("Survived", "mean"))
group_summary = group_summary[group_summary["passengers"] >= 10]

# Three highest survival rates
best3 = group_summary.sort_values("survival_rate", ascending=False).head(3)
# Three lowest survival rates
worst3 = group_summary.sort_values("survival_rate", ascending=True).head(3)
print("Top 3 groups:")
print(best3)
print("\nBottom 3 groups:")
print(worst3)

Top 3 groups:
                        passengers  survival_rate
Sex    Pclass AgeGroup                           
female 1      adult             83       0.975904
       2      adult             64       0.906250
       3      teen              12       0.666667

Bottom 3 groups:
                      passengers  survival_rate
Sex  Pclass AgeGroup                           
male 3      teen              18       0.055556
     2      adult             93       0.075269
     3      adult            300       0.123333
